# Sinmate: QLoRA Fine-Tuning Gemma-3-1B-IT for Spoken Sinhala

This Google Colab notebook fine-tunes **`google/gemma-3-1b-it`** using **4-bit QLoRA** on a conversational Sinhala dataset (*Kathitha Bhashawa* with Sri Lankan fillers, Singlish, and 4 personas).

### Pipeline Overview:
1. **Load Base Model**: `google/gemma-3-1b-it` quantized to 4-bit (VRAM footprint ~1.8GB on free Colab T4).
2. **Chat Template**: Standard Gemma turn tags `<start_of_turn>user...<end_of_turn>`.
3. **Train with QLoRA**: LoRA adapters on linear attention and MLP layers (`r=16, alpha=32`).
4. **Inference Test**: Multi-persona conversational test in colloquial Sinhala.
5. **Export GGUF**: Export 4-bit quantized GGUF weights (~700MB–800MB) for offline laptop execution.

In [6]:
# 1. Install fine-tuning dependencies
!pip install -q transformers datasets accelerate peft trl bitsandbytes sentencepiece

In [7]:
# 2. Hugging Face Login (Access Gemma license)
# Token lookup order: Colab secret -> HF_TOKEN env var -> getpass prompt
import os
from getpass import getpass
from huggingface_hub import login

token = None
try:
    from google.colab import userdata
    token = userdata.get('HF_TOKEN')
except Exception:
    pass  # not on Colab, or no HF_TOKEN secret / notebook access

token = token or os.environ.get('HF_TOKEN') or getpass('Enter your Hugging Face access token: ')
login(token=token)

In [8]:
# 3. Clone or upload Sinmate dataset
import os
if not os.path.exists("data"):
    !git clone https://github.com/chathuwa-whiz/Sinmate.git
    %cd Sinmate

from datasets import load_dataset
dataset = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})
print("Dataset loaded:", dataset)

Cloning into 'Sinmate'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (54/54), done.
remote: Total 59 (delta 1), reused 59 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 363.41 KiB | 2.86 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/Sinmate


Generating train split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['id', 'persona', 'input_type', 'category', 'messages'],
        num_rows: 108
    })
    val: Dataset({
        features: ['id', 'persona', 'input_type', 'category', 'messages'],
        num_rows: 12
    })
})


In [13]:
# 4. Load Gemma-3-1B-IT in 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_ID = "google/gemma-3-1b-it"

# bf16 needs compute capability >= 8.0 (A100/L4/...). The free-tier T4 is 7.5: bf16 kernels fail there,
# and Gemma 3 overflows in fp16, so fall back to fp32 compute (fine for a 1B model).
USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
COMPUTE_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
print(f"GPU: {torch.cuda.get_device_name()} | compute dtype: {COMPUTE_DTYPE}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=COMPUTE_DTYPE
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

GPU: Tesla T4 | compute dtype: torch.float32


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879


In [14]:
# 5. Format Chat Templates & Train
from trl import SFTConfig, SFTTrainer

def format_prompts(example):
    formatted = tokenizer.apply_chat_template(example["messages"], tokenize=False)
    return {"text": formatted}

# Keep only the "text" column so SFTTrainer doesn't pick up the raw "messages" field
formatted_dataset = dataset.map(format_prompts, remove_columns=dataset["train"].column_names)

training_args = SFTConfig(
    output_dir="./gemma3_sinhala_output",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=0.05,  # float < 1 = ratio of total steps (replaces warmup_ratio)
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=10,
    num_train_epochs=3,
    fp16=False,
    bf16=USE_BF16,  # set in cell 4; False on T4 (fp32 compute)
    save_strategy="epoch",
    eval_strategy="epoch",
    dataset_text_field="text",
    max_length=512
)

# The model is already a PEFT model (cell 4), so no peft_config here
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["val"],
    processing_class=tokenizer,
    args=training_args
)

trainer.train()

Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,5.073893,2.087605,2.068345,0.615474,14820.000000
2,1.952161,0.969160,0.966688,0.815101,29640.000000
3,0.664136,0.751785,0.785327,0.849940,44460.000000


TrainOutput(global_step=42, training_loss=2.1177807450294495, metrics={'train_runtime': 166.8562, 'train_samples_per_second': 1.942, 'train_steps_per_second': 0.252, 'total_flos': 203940771632640.0, 'train_loss': 2.1177807450294495, 'epoch': 3.0})

In [15]:
# 6. Test Interactive Multi-Persona Inference
test_messages = [
    {"role": "system", "content": "You are Sinmate, an affectionate native Sinhala companion.\n[ROLE: BEST_FRIEND]\nUse friendly colloquial Sinhala (machan), strictly 1-2 spoken sentences."},
    {"role": "user", "content": "Mata ada mara kammali ban, monada karanna hoda?"}
]

prompt = tokenizer.apply_chat_template(test_messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=64, temperature=0.7, top_p=0.9)

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
print("\n🤖 SINMATE RESPONSE:", response)


🤖 SINMATE RESPONSE: හරි මචි, ගහ ගහ. ඒක හරිම හොඳ උනන්දුව.


In [16]:
# 7. Save LoRA Adapters
model.save_pretrained("sinmate-gemma-3-1b-lora")
tokenizer.save_pretrained("sinmate-gemma-3-1b-lora")
print("Adapters saved successfully to sinmate-gemma-3-1b-lora!")

Adapters saved successfully to sinmate-gemma-3-1b-lora!


In [ ]:
from google.colab import drive
drive.mount('/content/drive')